In [6]:
import numpy as np
import pandas as pd
import os

#import microsim modules that we need in order to run a trial
from microsim.trials.trial_type import TrialType
from microsim.trials.trial_description import NhanesTrialDescription
from microsim.risk_factors.risk_factor import StaticRiskFactorsType, DynamicRiskFactorsType
from microsim.person.person_filter_factory import PersonFilterFactory
from microsim.trials.trial_outcome_assessor_factory import TrialOutcomeAssessorFactory
from microsim.trials.trial import Trial
from microsim.population.population_factory import PopulationFactory
from microsim.outcomes.outcome import OutcomeType
from microsim.person.person_filter import PersonFilter
from microsim.person.person_factory import PersonFactory

#switch to the Microsim directory
microsimDir = "/Users/deligkaris.1/OneDrive - The Ohio State University Wexner Medical Center/MICROSIM/CODE/microsim"
os.chdir(microsimDir)

Specify NHANES population variables

In [8]:
year=2017
duration=1
durationEndOfTime=200
nhanesWeights=True
popSize=100000
nWorkers=5
riskScaling = {OutcomeType.EPILEPSY: 1.}
riskScalingPrevalence = {OutcomeType.EPILEPSY: 2}
#riskScaling = None

### Incidence and prevalence

Incidence and prevalence right after the creation of the population

In [ ]:
pop = PopulationFactory.get_nhanes_population(n=popSize, year=year, nhanesWeights=nhanesWeights, riskScaling=riskScaling, prevalenceRiskScaling=riskScalingPrevalence)

pop.print_outcome_incidence_prevalence(outcomeType=OutcomeType.EPILEPSY)

Incidence and prevalence after a year

In [ ]:
pop.advance(years=duration, nWorkers=nWorkers)

pop.print_outcome_incidence_prevalence(outcomeType=OutcomeType.EPILEPSY)

### Epilepsy population characteristics at baseline (using prevalence epilepsy model only)

In [9]:
pf = PersonFilter()
pf.add_filter("person", "epilepsy", lambda x: x.has_outcome(OutcomeType.EPILEPSY, inSim=False))

popSize = 10000

pop = PopulationFactory.get_nhanes_population(n=popSize, year=year, nhanesWeights=nhanesWeights, personFilters=pf, riskScaling=riskScaling, prevalenceRiskScaling=riskScalingPrevalence)

pop.print_baseline_summary()

                          Printing a summary at baseline...
                          Printing a summary of risk factors and default treatments...
                          min      0.25    med     0.75     max    mean     sd
                          -----------------------------------------------------
                    age     0.0    10.0    31.0    59.0    80.0    34.6    25.6
                    sbp    72.0   103.3   114.0   129.3   238.0   118.7    21.5
                    dbp    40.0    58.0    66.0    75.3   124.7    67.0    13.1
                    a1c     3.8     5.1     5.4     5.8    16.2     5.7     1.0
                    hdl    10.0    44.0    53.0    64.0   189.0    54.9    15.5
                    ldl    18.0    80.0   100.0   124.0   375.0   104.3    34.9
                   trig    13.0    54.0    79.0   118.0  2684.0    97.7    82.7
                totChol    79.0   148.0   172.0   200.0   446.0   177.2    40.6
                    bmi    12.3    19.1    25.2    31.

### Epilepsy adult lifetime risk

In [ ]:
pf = PersonFilter()
pf.add_filter("df", "18yo", lambda x: x[PersonFactory.microsimToNhanes[DynamicRiskFactorsType.AGE.value]]==18)

pop = PopulationFactory.get_nhanes_population(n=popSize, year=year, nhanesWeights=nhanesWeights, personFilters=pf, riskScaling=riskScaling, prevalenceRiskScaling=riskScalingPrevalence)

pop.print_baseline_summary()

pop.advance(years=durationEndOfTime, nWorkers=nWorkers)

pop.print_outcome_incidence_prevalence(outcomeType=OutcomeType.EPILEPSY)

### Global burden of disease idiopathic epilepsy new cases

In [ ]:
#https://vizhub.healthdata.org/gbd-compare/
#advanced settings, choose epilepsy as the cause ad measure either incidence or prevalence

newCasesGBD = {"15-19": 43.78, #new cases per 100,000
               "20-24": 37.43,
               "25-29": 32.37,
               "30-34": 28.43,
               "35-39": 26.08,
               "40-44": 24.61,
               "45-49": 22.87,
               "50-54": 22.8,
               "55-59": 24.65,
               "60-64": 26.11,
               "65-69": 24.71,
               "70-74": 26.66,
               "75-79": 32.53,
               "80-84": 39.28,  
               "85-89": 46.61,  
               "90-94": 60.85,  
               "95-99": 85.17}

print(" "*19, "age", "  rate")
for group, rate in newCasesGBD.items():
            print(f"{group:>23} {rate/100000:6.5f}")

### Global burden of disease idiopathic epilepsy prevalence

In [ ]:
prevalenceGBD = {"15-19": 350.08, #prevalence rates per 100,000
                 "20-24": 338.18,
                 "25-29": 314.85,
                 "30-34": 285.74,
                 "35-39": 264.1,
                 "40-44": 254.15,
                 "45-49": 241.94,
                 "50-54": 238.33,
                 "55-59": 263.56,
                 "60-64": 322.5,
                 "65-69": 372.46,
                 "70-74": 418.66,
                 "75-79": 483.75,
                 "80-84": 561.57,  
                 "85-89": 649.94,  
                 "90-94": 767.18,  
                 "95-99": 951.51}

print(" "*19, "age", "  rate")
for group, rate in prevalenceGBD.items():
            print(f"{group:>23} {rate/100000:6.5f}")

### Trial with a US sample

In [ ]:
%%time

pf = None
td = NhanesTrialDescription(trialType = TrialType.COMPLETELY_RANDOMIZED,
                            blockFactors=list(),
                            sampleSize = popSize, 
                            duration = duration, 
                            treatmentStrategies = "sprint",
                            nWorkers = nWorkers, 
                            personFilters=pf, 
                            year=year, nhanesWeights=nhanesWeights, distributions=False)

tr = Trial(td)

tr.print_covariate_distributions()

tr.run()
toa = TrialOutcomeAssessorFactory.get_trial_outcome_assessor()
tr.analyze(toa)
print(tr)

tr.print_treatment_strategy_variables_information()

### Trial with the US epilepsy sample

In [10]:
%%time

pf = PersonFilter()
pf.add_filter("person", "epilepsy", lambda x: x.has_outcome(OutcomeType.EPILEPSY))

td = NhanesTrialDescription(trialType = TrialType.COMPLETELY_RANDOMIZED,
                            blockFactors=list(),
                            sampleSize = popSize, 
                            duration = duration, 
                            treatmentStrategies = "sprint",
                            nWorkers = nWorkers, 
                            personFilters=pf, 
                            year=year, nhanesWeights=nhanesWeights, distributions=False)

tr = Trial(td)

tr.print_covariate_distributions()

tr.run()
toa = TrialOutcomeAssessorFactory.get_trial_outcome_assessor()
tr.analyze(toa)
print(tr)

tr.print_treatment_strategy_variables_information()

KeyboardInterrupt: 